# UltraChat SFT — chat with a finetuned checkpoint

Loads an SFT checkpoint produced by `train.py` and generates replies via the
ChatML template. The architecture flags below **must match the run** (the
checkpoint stores no model hyperparameters — pull them from the run's wandb
config if unsure).

In [ ]:
import os

os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")

import torch

from chimera.data import UltraChatDataModule
from chimera.models import GPT

CKPT = "/mnt/ai/runs/fineweb-edu/sft/<run-name>/checkpoints/sft.ckpt"

# architecture of the checkpointed run (defaults of train.py shown)
ARCH = dict(n_embd=384, n_head=12, n_kv_head=3, n_layer=6)
SEQ_LEN = 2048

In [ ]:
# tokenizer + ChatML rendering come from the DataModule (no dataset download:
# setup() is skipped, only the tokenizer is loaded)
dm = UltraChatDataModule(data_dir="/mnt/ai/data", seq_len=SEQ_LEN)
dm.tokenizer = dm._load_tokenizer()
dm.vocab_size = dm.tokenizer.vocab_size
dm.eos_id = dm._special_id(dm.eos_token)
dm.im_start_id = dm._special_id(dm.im_start_token)
dm.im_end_id = dm._special_id(dm.im_end_token)

model = GPT(vocab_size=dm.vocab_size, block_size=SEQ_LEN, **ARCH)
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
cleaned = {}
for k, v in ckpt["state_dict"].items():
    for prefix in ("model.", "_orig_mod."):
        if k.startswith(prefix):
            k = k[len(prefix):]
    cleaned[k] = v
model.load_state_dict(cleaned, strict=True)
# Lightning checkpoints load on CPU -- move to GPU or generation is ~30x slower
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval().to(device)
print(f"loaded {CKPT} (global_step={ckpt.get('global_step')}) on {device}")

In [ ]:
@torch.no_grad()
def chat(prompt: str, max_new_tokens: int = 256, temperature: float = 0.7) -> str:
    ids = dm.render_prompt([{"role": "user", "content": prompt}])
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature)
    out = out[0, len(ids):].tolist()
    if dm.im_end_id in out:  # trim at the assistant's end-of-turn token
        out = out[: out.index(dm.im_end_id)]
    return dm.decode(out)


print(chat("What is photosynthesis?"))